# Refinement
Fonction qui va permettre de refine le topolical graphe initial, avec marge d'erreur de 2px

In [1]:
import networkx as nx
import numpy as np
from typing import List, Optional

In [ ]:
def fit_bezier(pixel_c : np.array):
    return param, error


def devide(pixel_c: np.ndarray, precision: float = 2.0) -> Optional[int]:
    _, e = fit_bezier(pixel_c)
    if e <= precision:
        return None

    N = len(pixel_c)

    _, e1 = fit_bezier(pixel_c[:N // 3])
    _, e2 = fit_bezier(pixel_c[N // 3:])
    _, e3 = fit_bezier(pixel_c[:2 * N // 3])
    _, e4 = fit_bezier(pixel_c[2 * N // 3:])

    if e1 + e2 < e3 + e4:
        first = devide(pixel_c[:N//3])
        second = devide(pixel_c[N//3:])
        idx = np.array([N//3])

        if first is None:
            first = np.array([])
        if second is None:
            second = np.array([])
        else:
            second = second + idx  # ajout de idx à chaque élément

        return np.concatenate([first, idx, second])
    
    else : 
        first = devide(pixel_c[:2*N//3])
        second = devide(pixel_c[2*N//3:])
        idx = np.array([2*N//3])

        if first is None:
            first = np.array([])
        if second is None:
            second = np.array([])
        else:
            second = second + idx  # ajout de idx à chaque élément

        return np.concatenate([first, idx, second])



def refine(topo_g: nx.MultiDiGraph, topo_c: List[np.ndarray], precision: float = 2.0):
    new_edges = []
    new_curves = []

    next_node_id = max(topo_g.nodes) + 1 if topo_g.nodes else 0

    for u, v, key in topo_g.edges(keys=True):
        curve = topo_c[key]
        indices = devide(curve, precision)

        if indices is None or len(indices) == 0:
            # Rien à changer
            new_edges.append((u, v))
            new_curves.append(curve)
            continue

        # Supprimer l’arête originale
        topo_g.remove_edge(u, v, key=key)

        # Créer les nouveaux points d'interpolation
        indices = list(indices.astype(int))
        cuts = [0] + indices + [len(curve)]
        points = [curve[cuts[i]:cuts[i+1]] for i in range(len(cuts) - 1)]

        # Ajouter les nouveaux nœuds et arêtes
        nodes = [u] + [next_node_id + i for i in range(len(points) - 2)] + [v]
        for i in range(len(points)):
            if i < len(points) - 1:
                new_edges.append((nodes[i], nodes[i+1]))
                new_curves.append(points[i])
        next_node_id += len(points) - 2

    # Remplacer topo_c par les courbes mises à jour
    topo_g.clear_edges()
    for i, (a, b) in enumerate(new_edges):
        topo_g.add_edge(a, b, key=i)
    topo_c.clear()
    topo_c.extend(new_curves)


